In [ ]:
# Exercise 1: Pretrained sentiment analysis
# Sources: Lab 3/docs/08_implementation.md; Lab 3/docs/03_model.md; Lab 3/docs/07_demo.md
from pathlib import Path
import sys
ROOT = Path('Lab 3') if Path('Lab 3').is_dir() else Path('.')
sys.path.insert(0, str(ROOT.resolve()))
from notebook_helpers import infer_and_report
text = 'Bộ phim này rất hay!'
infer_and_report(text)


In [ ]:
# Exercise 2 - Step 1: Prepare libraries
# Sources: Lab 3/docs/08_implementation.md; Lab 3/docs/10_sources.md
from pathlib import Path
import sys
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)

ROOT = Path('Lab 3') if Path('Lab 3').is_dir() else Path('.')
sys.path.insert(0, str(ROOT.resolve()))
from notebook_helpers import available_checkpoint, train_and_save, show_training_time, evaluate_and_report
MODEL_NAME = 'distilbert/distilbert-base-uncased'
set_seed(42)
print('Thiết bị:', 'CUDA' if torch.cuda.is_available() else 'CPU')


In [ ]:
# Exercise 2 - Step 2: Load and split IMDB
# Sources: Lab 3/docs/02_data.md; Lab 3/docs/08_implementation.md
raw_train, raw_test = load_dataset('stanfordnlp/imdb', split=['train', 'test'])
train_ids, val_ids = train_test_split(np.arange(len(raw_train)), test_size=0.2,
                                     stratify=raw_train['label'], random_state=42)
data = DatasetDict(train=raw_train.select(train_ids),
                   validation=raw_train.select(val_ids), test=raw_test)
print({name: len(split) for name, split in data.items()})


In [ ]:
# Exercise 2 - Step 3: Load pretrained model and tokenizer
# Sources: Lab 3/docs/03_model.md; Lab 3/docs/04_training.md; Lab 3/docs/08_implementation.md
TRAIN = False
CHECKPOINT = available_checkpoint(ROOT) if not TRAIN else None
OUTPUT = ROOT/'outputs/simple-training'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME if TRAIN else str(CHECKPOINT))
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME if TRAIN else str(CHECKPOINT), num_labels=2,
    id2label={0: 'NEGATIVE', 1: 'POSITIVE'}, label2id={'NEGATIVE': 0, 'POSITIVE': 1})
print('Checkpoint:', CHECKPOINT)
print('Chế độ:', 'Huấn luyện mới' if TRAIN else 'Nạp model đã huấn luyện')


In [ ]:
# Exercise 2 - Step 4: Tokenize datasets
# Sources: Lab 3/docs/02_data.md; Lab 3/docs/03_model.md
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

tokenized = data.map(tokenize, batched=True, remove_columns=['text'])
collator = DataCollatorWithPadding(tokenizer)


In [ ]:
# Exercise 2 - Step 5: Configure training arguments
# Sources: Lab 3/docs/04_training.md; Lab 3/docs/08_implementation.md
args = TrainingArguments(
    output_dir=str(OUTPUT), eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=3, weight_decay=0.01, max_grad_norm=1.0, optim='adamw_torch',
    load_best_model_at_end=True, metric_for_best_model='accuracy', greater_is_better=True,
    save_total_limit=2, fp16=torch.cuda.is_available(), report_to='none',
    seed=42, logging_steps=50)


In [ ]:
# Exercise 2 - Step 6: Define metrics and train
# Sources: Lab 3/docs/05_evaluation.md; Lab 3/docs/04_training.md; Lab 3/docs/08_implementation.md; Lab 3/docs/11_AI_Usage.md

def compute_metrics(prediction):
    logits, labels = prediction
    predicted = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicted, average='binary', zero_division=0)
    return {'accuracy': accuracy_score(labels, predicted), 'precision': precision,
            'recall': recall, 'f1_binary': f1,
            'f1_macro': f1_score(labels, predicted, average='macro', zero_division=0)}

trainer = Trainer(
    model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['validation'],
    processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.01)])
if TRAIN:
    CHECKPOINT = train_and_save(trainer, tokenizer, OUTPUT)
show_training_time(CHECKPOINT)


In [ ]:
# Exercise 2 - Step 7: Evaluate and demonstrate
# Sources: Lab 3/docs/05_evaluation.md; Lab 3/docs/06_verification.md; Lab 3/docs/07_demo.md; Lab 3/docs/03_model.md
result = evaluate_and_report(trainer, tokenized['test'], CHECKPOINT, ROOT, TRAIN,
    sentences=['I loved this movie. The acting was great!', 'The movie was boring and disappointing.'])
